In [2]:
import os
import pandas as pd
import ollama
import re
from tqdm import tqdm

In [5]:
import os
import glob
import pandas as pd

def is_safely_binary(series):
    """
    Максимально аккуратная проверка. Удаляет столбец ТОЛЬКО если он
    гарантированно содержит бинарный код (картинку).
    """
    # Берем первые 50 непустых строк для оценки
    sample_data = series.dropna().astype(str).head(50)
    if sample_data.empty:
        return False
        
    text = "".join(sample_data)
    if not text:
        return False

    # 1. Сигнатуры файлов картинок (JPEG, PNG, GIF, WebP)
    signatures =['JFIF', 'IHDR', 'GIF89a', 'WEBP', 'Exif']
    if any(sig in text for sig in signatures):
        return True
        
    # 2. Наличие нулевого байта (железный признак сырых бинарных данных)
    if '\x00' in text:
        return True
        
    # 3. Наличие непечатных системных байтов (ASCII < 32).
    # Пропускаем переносы строк (\n, \r) и табы (\t), так как они бывают в нормальном тексте.
    control_chars = sum(1 for c in text if ord(c) < 32 and c not in ('\n', '\r', '\t'))
    
    # Если системного мусора больше 5%, это точно бинарник
    if len(text) > 0 and (control_chars / len(text)) > 0.05:
        return True

    # В любом другом случае (даже если это длинный ID или текст на другом языке) — оставляем
    return False

def clean_csvs_safe(input_folder, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    csv_files = glob.glob(os.path.join(input_folder, "*.csv"))

    if not csv_files:
        print(f"В папке {input_folder} не найдено CSV файлов.")
        return

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_folder, file_name)
        
        try:
            # Читаем CSV. 
            # encoding_errors='replace' не даст скрипту упасть при чтении бинарника
            df = pd.read_csv(
                file_path, 
                encoding='utf-8', 
                encoding_errors='replace',
                on_bad_lines='skip',
                engine='python'
            )
            
            # Ищем колонки с картинками по новым безопасным правилам
            cols_to_drop = [col for col in df.columns if is_safely_binary(df[col])]
            
            if cols_to_drop:
                df.drop(columns=cols_to_drop, inplace=True)
                print(f"[ОЧИЩЕНО] {file_name} -> удалены: {', '.join(cols_to_drop)}")
            else:
                print(f"[ОК] {file_name} -> мусорных колонок не найдено.")
                
            df.to_csv(output_path, index=False)
            
        except Exception as e:
            print(f"[ОШИБКА] Файл {file_name}: {e}")


In [ ]:
# base_dir = "set/"

# for cluster in os.listdir(base_dir):
#     cluster_dir = os.path.join(base_dir, cluster)
#     print(f"PROCESSING {cluster}")

#     for index in range(1, 11):
#         my_input = os.path.join(cluster_dir, str(index), "csv")
#         my_output = os.path.join(cluster_dir, str(index), "csv_russian")
#         my_model = "aya:8b"

#         clean_csvs_safe(my_input, my_input)

PROCESSING pubs
[ОК] titles.csv -> мусорных колонок не найдено.
[ОК] employee.csv -> мусорных колонок не найдено.
[ОК] titleauthor.csv -> мусорных колонок не найдено.
[ОК] stores.csv -> мусорных колонок не найдено.
[ОК] roysched.csv -> мусорных колонок не найдено.
[ОК] authors.csv -> мусорных колонок не найдено.
[ОК] sales.csv -> мусорных колонок не найдено.
[ОЧИЩЕНО] pub_info.csv -> удалены: logo
[ОК] jobs.csv -> мусорных колонок не найдено.
[ОК] publishers.csv -> мусорных колонок не найдено.
[ОК] discounts.csv -> мусорных колонок не найдено.
[ОК] titles.csv -> мусорных колонок не найдено.
[ОК] employee.csv -> мусорных колонок не найдено.
[ОК] titleauthor.csv -> мусорных колонок не найдено.
[ОК] stores.csv -> мусорных колонок не найдено.
[ОК] roysched.csv -> мусорных колонок не найдено.
[ОК] authors.csv -> мусорных колонок не найдено.
[ОК] sales.csv -> мусорных колонок не найдено.
[ОЧИЩЕНО] pub_info.csv -> удалены: logo
[ОК] jobs.csv -> мусорных колонок не найдено.
[ОК] publishers.csv

In [ ]:
# import os
# import pandas as pd
# import ollama
# import re
# from tqdm import tqdm

# # --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

# def is_id_column(col_name):
#     """
#     Проверяет, содержит ли название колонки 'id', 'Id' или 'ID'.
#     """
#     return 'id' in col_name.lower()

# def is_translatable(text):
#     """
#     Проверяет, является ли содержимое ячейки текстом для перевода.
#     Исключает числа, даты, пустые строки.
#     """
#     if not isinstance(text, str):
#         return False
#     text = text.strip()
#     # Если пусто или состоит только из цифр/знаков препинания
#     if not text or re.match(r'^[\d\s\.,\-:;/]+$', text):
#         return False
#     return True

# def translate_text(text, model_name):
#     """
#     Отправляет запрос в Ollama.
#     """
#     try:
#         response = ollama.chat(model=model_name, messages=[
#             {
#                 'role': 'system',
#                 'content': (
#                     'You are a professional translator. Translate from English to Russian. '
#                     'Output ONLY the translation. '
#                     'Keep formatting, codes, and proper names if necessary. '
#                     'DO NOT OUTPUT ANY ADDITIONAL TEXT, TRANSLATION ONLY!'
#                 )
#             },
#             {
#                 'role': 'user',
#                 'content': text
#             },
#         ])
#         return response['message']['content'].strip()
#     except Exception as e:
#         print(f"\n[!] Ошибка Ollama при переводе '{text}': {e}")
#         return text

# # --- ОСНОВНАЯ ЛОГИКА ---

# def collect_unique_values(files, input_dir):
#     """
#     Проходит по всем файлам и собирает уникальные строки, требующие перевода.
#     """
#     unique_texts = set()
    
#     print(">>> ЭТАП 1: Сбор уникальных значений из всех таблиц...")
    
#     for filename in tqdm(files, desc="Сканирование файлов"):
#         full_path = os.path.join(input_dir, filename)
#         try:
#             df = pd.read_csv(full_path, keep_default_na=False)
            
#             # Отфильтровываем колонки с ID
#             target_cols = [c for c in df.columns if not is_id_column(c)]
            
#             for col in target_cols:
#                 # Берем уникальные значения из колонки
#                 values = df[col].dropna().unique()
                
#                 # Добавляем в сет только то, что проходит проверку is_translatable
#                 for val in values:
#                     if is_translatable(val):
#                         unique_texts.add(val)
                        
#         except Exception as e:
#             print(f"[!] Ошибка чтения {filename}: {e}")
            
#     return list(unique_texts)

# def build_translation_map(unique_texts, model_name):
#     """
#     Переводит список уникальных строк и создает словарь {English: Russian}.
#     """
#     translation_map = {}
    
#     print(f"\n>>> ЭТАП 2: Перевод уникальных значений (всего: {len(unique_texts)})...")
    
#     # tqdm показывает прогресс перевода
#     for text in tqdm(unique_texts, desc="Запросы к LLM"):
#         translated = translate_text(text, model_name)
#         translation_map[text] = translated
        
#     return translation_map

# def apply_translations(files, input_dir, output_dir, translation_map):
#     """
#     Применяет словарь переводов к файлам и сохраняет их.
#     """
#     print(f"\n>>> ЭТАП 3: Применение переводов и сохранение...")
    
#     for filename in tqdm(files, desc="Обработка таблиц"):
#         input_path = os.path.join(input_dir, filename)
#         output_path = os.path.join(output_dir, filename) # Имя файла то же, папка другая
        
#         try:
#             df = pd.read_csv(input_path, keep_default_na=False)
            
#             # Определяем колонки, которые НЕ являются ID
#             target_cols = [c for c in df.columns if not is_id_column(c)]
            
#             # Проходим только по нужным колонкам
#             for col in target_cols:
#                 # Функция map заменит значение, если оно есть в словаре, 
#                 # иначе оставит старое значение (x)
#                 df[col] = df[col].map(lambda x: translation_map.get(x, x))
            
#             df.to_csv(output_path, index=False, encoding='utf-8-sig')
            
#         except Exception as e:
#             print(f"[!] Ошибка обработки {filename}: {e}")

# def process_csv_directory(input_dir, output_dir, model="qwen2.5:7b"):
#     # Проверки путей
#     if not os.path.exists(output_dir):
#         os.makedirs(output_dir)
#         print(f"[+] Создана папка: {output_dir}")

#     files = [f for f in os.listdir(input_dir) if f.endswith('.csv')]
#     if not files:
#         print(f"[!] В директории '{input_dir}' CSV-файлы не найдены.")
#         return

#     print(f"[*] Найдено файлов: {len(files)}")
#     print(f"[*] Используемая модель: {model}")

#     # 1. Собираем уникальные тексты
#     unique_texts = collect_unique_values(files, input_dir)
#     print(f"[*] Найдено уникальных фраз для перевода: {len(unique_texts)}")

#     if not unique_texts:
#         print("[!] Не найдено текста для перевода. Завершение.")
#         return

#     # 2. Создаем словарь переводов
#     translation_map = build_translation_map(unique_texts, model)

#     # 3. Применяем словарь ко всем файлам
#     apply_translations(files, input_dir, output_dir, translation_map)
    
#     print("\n[V] Готово! Все файлы обработаны.")

In [1]:
import os
import pandas as pd
import re
import torch
from tqdm import tqdm
from transformers import MarianMTModel, MarianTokenizer

# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

def is_id_column(col_name):
    """
    Проверяет, содержит ли название колонки 'id', 'Id' или 'ID'.
    """
    return 'id' in col_name.lower()

def is_translatable(text):
    """
    Проверяет, является ли содержимое ячейки текстом для перевода.
    Исключает числа, даты, пустые строки.
    """
    if not isinstance(text, str):
        return False
    text = text.strip()
    # Если пусто или состоит только из цифр/знаков препинания
    if not text or re.match(r'^[\d\s\.,\-:;/]+$', text):
        return False
    return True

# --- ОСНОВНАЯ ЛОГИКА ---

def collect_unique_values(files, input_dir):
    """
    Проходит по всем файлам и собирает уникальные строки, требующие перевода.
    """
    unique_texts = set()
    
    print(">>> ЭТАП 1: Сбор уникальных значений из всех таблиц...")
    
    for filename in tqdm(files, desc="Сканирование файлов"):
        full_path = os.path.join(input_dir, filename)
        try:
            df = pd.read_csv(full_path, keep_default_na=False)
            
            target_cols =[c for c in df.columns if not is_id_column(c)]
            
            for col in target_cols:
                values = df[col].dropna().unique()
                for val in values:
                    if is_translatable(val):
                        unique_texts.add(val)
                        
        except Exception as e:
            print(f"[!] Ошибка чтения {filename}: {e}")
            
    return list(unique_texts)

def build_translation_map(unique_texts, tokenizer, model, device, batch_size=32):
    """
    Переводит список уникальных строк ПАКЕТАМИ (batches) для максимальной скорости 
    и создает словарь {English: Russian}.
    """
    translation_map = {}
    
    print(f"\n>>> ЭТАП 2: Перевод уникальных значений (всего: {len(unique_texts)})...")
    
    # Разбиваем список на батчи
    for i in tqdm(range(0, len(unique_texts), batch_size), desc="Пакетный перевод"):
        batch = unique_texts[i : i + batch_size]
        
        try:
            # Подготовка текста для модели
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            # Генерация перевода (детерминированная, do_sample=False по умолчанию)
            with torch.no_grad():
                translated_tokens = model.generate(**inputs)
            
            # Декодирование токенов обратно в текст
            decoded_translations =[
                tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens
            ]
            
            # Заполняем словарь
            for src, tgt in zip(batch, decoded_translations):
                translation_map[src] = tgt.strip()
                
        except Exception as e:
            print(f"\n[!] Ошибка перевода батча: {e}")
            # В случае ошибки оставляем оригинальный текст
            for src in batch:
                translation_map[src] = src
                
    return translation_map

def apply_translations(files, input_dir, output_dir, translation_map):
    """
    Применяет словарь переводов к файлам и сохраняет их.
    """
    print(f"\n>>> ЭТАП 3: Применение переводов и сохранение...")
    
    for filename in tqdm(files, desc="Обработка таблиц"):
        input_path = os.path.join(input_dir, filename)
        output_path = os.path.join(output_dir, filename)
        
        try:
            df = pd.read_csv(input_path, keep_default_na=False)
            target_cols =[c for c in df.columns if not is_id_column(c)]
            
            for col in target_cols:
                df[col] = df[col].map(lambda x: translation_map.get(x, x))
            
            df.to_csv(output_path, index=False, encoding='utf-8-sig')
            
        except Exception as e:
            print(f"[!] Ошибка обработки {filename}: {e}")

def process_csv_directory(input_dir, output_dir, batch_size=32):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"[+] Создана папка: {output_dir}")

    files =[f for f in os.listdir(input_dir) if f.endswith('.csv')]
    if not files:
        print(f"[!] В директории '{input_dir}' CSV-файлы не найдены.")
        return

    print(f"[*] Найдено файлов: {len(files)}")
    
    # Инициализация легковесной модели перевода (Загружается 1 раз)
    model_name = "Helsinki-NLP/opus-mt-en-ru"
    print(f"[*] Загрузка модели {model_name}...")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Используется устройство: {device}")
    
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device)

    # 1. Собираем уникальные тексты
    unique_texts = collect_unique_values(files, input_dir)
    print(f"[*] Найдено уникальных фраз для перевода: {len(unique_texts)}")

    if not unique_texts:
        print("[!] Не найдено текста для перевода. Завершение.")
        return

    # 2. Создаем словарь переводов
    translation_map = build_translation_map(unique_texts, tokenizer, model, device, batch_size)

    # 3. Применяем словарь ко всем файлам
    apply_translations(files, input_dir, output_dir, translation_map)
    
    print("\n[V] Готово! Все файлы обработаны.")

/home/george/anaconda3/envs/hybrid_graphs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
base_dir = "set/"

for cluster in os.listdir(base_dir):
    cluster_dir = os.path.join(base_dir, cluster)
    print(f"PROCESSING {cluster}")

    for index in range(1, 11):
        my_input = os.path.join(cluster_dir, str(index), "csv")
        my_output = os.path.join(cluster_dir, str(index), "csv_russian_2")

        process_csv_directory(my_input, my_output)

In [3]:
import os
os.listdir("set")

['pubs',
 'northwind',
 'NCAA',
 'lahman_2014',
 'sakila',
 'Hockey',
 'ErgastF1',
 'Credit',
 'Mondial',
 'Basketball_men']

In [3]:
import os
import json
from tqdm import tqdm
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# --- 1. ФУНКЦИЯ ПЕРЕВОДА (ЧЕРЕЗ LANGCHAIN + OLLAMA) ---

def build_translation_map(unique_texts, chain, batch_size=10):
    """
    Переводит список уникальных строк батчами с помощью LangChain.
    Создает словарь {English: Russian}.
    """
    translation_map = {}
    if not unique_texts:
        return translation_map
        
    unique_list = list(unique_texts)
    print(f"    Перевод {len(unique_list)} уникальных вопросов...")
    
    # Пакетная обработка через LangChain batch
    # Используем небольшой batch_size, так как LLM работает медленнее классических переводчиков
    for i in tqdm(range(0, len(unique_list), batch_size), desc="    Батчи", leave=False):
        batch = unique_list[i : i + batch_size]
        try:
            # Формируем список словарей для LangChain
            inputs = [{"text": text} for text in batch]
            
            # Генерация перевода (работает параллельно/последовательно в зависимости от настроек Ollama)
            results = chain.batch(inputs)
            
            for src, tgt in zip(batch, results):
                # Очищаем ответ от случайных кавычек, переносов строк и пробелов (LLM иногда их добавляют)
                cleaned_tgt = tgt.strip(" \n\"'")
                translation_map[src] = cleaned_tgt
                
        except Exception as e:
            print(f"\n[!] Ошибка перевода батча: {e}")
            for src in batch:
                translation_map[src] = src
                
    return translation_map

# --- 2. ФУНКЦИЯ ОБРАБОТКИ КОНКРЕТНОГО ФАЙЛА ---

def process_qa_file(input_dir, output_dir, chain, batch_size=10):
    # Жестко задаем имена файлов
    input_filename = "QA_english.json"
    output_filename = "QA_russian.json"
    
    input_path = os.path.join(input_dir, input_filename)
    
    # Если нужного файла нет в папке, просто пропускаем эту папку
    if not os.path.exists(input_path):
        return

    # Создаем папку для сохранения, если ее нет
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    output_path = os.path.join(output_dir, output_filename)
    print(f"  > Нашли файл! Обработка: {input_path} -> {output_filename}")

    unique_questions = set()
    data = None
    
    # 1. Читаем QA_english.json и собираем вопросы
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Обработка, если в файле список:[{...}, {...}]
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict) and "question" in item:
                        unique_questions.add(item["question"])
            # Обработка, если в файле один объект: {...}
            elif isinstance(data, dict):
                if "question" in data:
                    unique_questions.add(data["question"])
                    
    except Exception as e:
        print(f"    [!] Ошибка чтения {input_filename}: {e}")
        return

    if not unique_questions:
        print("    [!] Вопросы для перевода не найдены.")
        return

    # 2. Создаем словарь переводов через LangChain
    translation_map = build_translation_map(unique_questions, chain, batch_size)

    # 3. Заменяем текст в данных на русский и сохраняем в QA_russian.json
    try:
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and "question" in item:
                    item["question"] = translation_map.get(item["question"], item["question"])
        elif isinstance(data, dict):
            if "question" in data:
                data["question"] = translation_map.get(data["question"], data["question"])

        # Сохраняем переведенный файл
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
            
    except Exception as e:
        print(f"    [!] Ошибка сохранения {output_filename}: {e}")


# --- 3. ГЛАВНЫЙ ЦИКЛ (ЗАПУСК) ---

if __name__ == "__main__":
    base_dir = "set/"
    
    # ------------------------------------------------------------------
    # Инициализируем LangChain Pipeline (Aya-8b через Ollama)
    # ------------------------------------------------------------------
    print("[*] Подключение к Ollama (модель: aya:8b)...")
    
    # temperature=0.0 заставляет модель быть детерминированной (без креативности)
    llm = Ollama(model="aya:8b", temperature=0.0)
    
    # Строгий промпт, чтобы избежать "мусора" в ответах (например: "Вот ваш перевод:")
    prompt = PromptTemplate(
        input_variables=["text"],
        template=(
            "You are a professional translator. Translate the following English question into Russian.\n"
            "Provide ONLY the translated text without any quotes, explanations, or conversational filler.\n\n"
            "English: {text}\n"
            "Russian:"
        )
    )
    
    # Создаем цепочку (Chain) через LCEL
    chain = prompt | llm
    print("[*] Модель успешно подключена!\n")
    # ------------------------------------------------------------------

    # Проходим по всем кластерам
    for cluster in os.listdir(base_dir):
        cluster_dir = os.path.join(base_dir, cluster)
        
        if not os.path.isdir(cluster_dir):
            continue
            
        print(f"\n{'='*40}\nPROCESSING CLUSTER: {cluster}\n{'='*40}")

        for index in range(1, 11):
            index_dir = os.path.join(cluster_dir, str(index))
            if not os.path.isdir(index_dir):
                continue
                
            # Указываем папки, где искать и куда сохранять (в ту же папку index_dir)
            my_input = index_dir      
            my_output = index_dir

            # Запускаем целевую обработку
            process_qa_file(
                input_dir=my_input, 
                output_dir=my_output, 
                chain=chain, 
                batch_size=5  # Для LLM лучше использовать небольшие батчи (5-10)
            )

/home/george/anaconda3/envs/hybrid_graphs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_140967/1897755492.py:122: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="aya:8b", temperature=0.0)


[*] Подключение к Ollama (модель: aya:8b)...
[*] Модель успешно подключена!


PROCESSING CLUSTER: pubs
  > Нашли файл! Обработка: set/pubs/1/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/pubs/2/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/pubs/3/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/pubs/4/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


...

  > Нашли файл! Обработка: set/Basketball_men/6/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/Basketball_men/7/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/Basketball_men/8/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/Basketball_men/9/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


  > Нашли файл! Обработка: set/Basketball_men/10/QA_english.json -> QA_russian.json
    Перевод 1 уникальных вопросов...


In [4]:
import os
import json
import re
import torch
from tqdm import tqdm
from transformers import MarianMTModel, MarianTokenizer

# --- 1. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ---

def is_translatable(text):
    """
    Проверяет, является ли содержимое текста пригодным для перевода.
    Исключает числа, даты, пустые строки (например, '18.5833' будет пропущено).
    """
    # Если это число (int/float) в JSON, пропускаем
    if not isinstance(text, str):
        return False
        
    text = text.strip()
    # Если пусто или состоит только из цифр и знаков препинания
    if not text or re.match(r'^[\d\s\.,\-:;/]+$', text):
        return False
        
    return True

def build_translation_map(unique_texts, tokenizer, model, device, batch_size=32):
    """Переводит список уникальных строк батчами и создает словарь {English: Russian}."""
    translation_map = {}
    if not unique_texts:
        return translation_map
        
    unique_list = list(unique_texts)
    print(f"    Перевод {len(unique_list)} уникальных текстовых ответов...")
    
    for i in tqdm(range(0, len(unique_list), batch_size), desc="    Батчи", leave=False):
        batch = unique_list[i : i + batch_size]
        try:
            inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            
            with torch.no_grad():
                translated_tokens = model.generate(**inputs)
            
            decoded_translations =[
                tokenizer.decode(t, skip_special_tokens=True) for t in translated_tokens
            ]
            
            for src, tgt in zip(batch, decoded_translations):
                translation_map[src] = tgt.strip()
                
        except Exception as e:
            print(f"\n[!] Ошибка перевода батча: {e}")
            for src in batch:
                translation_map[src] = src
                
    return translation_map

# --- 2. ФУНКЦИЯ ОБРАБОТКИ ОТВЕТОВ В ФАЙЛЕ ---

def process_qa_answers_file(input_dir, output_dir, tokenizer, model, device, batch_size=32):
    # Сначала ищем QA_russian.json (чтобы не затереть переведенные вопросы)
    # Если его нет, берем QA_english.json
    input_path = os.path.join(input_dir, "QA_russian.json")
    if not os.path.exists(input_path):
        input_path = os.path.join(input_dir, "QA_english.json")
        
    if not os.path.exists(input_path):
        return

    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    output_filename = "QA_russian.json"
    output_path = os.path.join(output_dir, output_filename)
    print(f"  > Чтение файла: {input_path}")

    unique_answers = set()
    data = None
    
    # 1. Читаем файл и собираем ОТВЕТЫ, пригодные для перевода
    try:
        with open(input_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict) and "answer" in item:
                        ans = item["answer"]
                        if is_translatable(ans):
                            unique_answers.add(ans)
                            
            elif isinstance(data, dict):
                if "answer" in data:
                    ans = data["answer"]
                    if is_translatable(ans):
                        unique_answers.add(ans)
                        
    except Exception as e:
        print(f"    [!] Ошибка чтения {input_path}: {e}")
        return

    if not unique_answers:
        print("    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).")
        # Сохраняем файл как есть, если мы брали его из english
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        return

    # 2. Создаем словарь переводов для ответов
    translation_map = build_translation_map(unique_answers, tokenizer, model, device, batch_size)

    # 3. Заменяем текст ответов на русский
    try:
        if isinstance(data, list):
            for item in data:
                if isinstance(item, dict) and "answer" in item:
                    ans = item["answer"]
                    if is_translatable(ans):
                        item["answer"] = translation_map.get(ans, ans)
                        
        elif isinstance(data, dict):
            if "answer" in data:
                ans = data["answer"]
                if is_translatable(ans):
                    data["answer"] = translation_map.get(ans, ans)

        # Сохраняем обновленный файл
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"    [V] Ответы переведены и сохранены в {output_filename}")
            
    except Exception as e:
        print(f"    [!] Ошибка сохранения {output_filename}: {e}")


# --- 3. ГЛАВНЫЙ ЦИКЛ (ЗАПУСК) ---

if __name__ == "__main__":
    base_dir = "set/"
    
    # Загружаем модель 1 раз перед циклом
    model_name = "Helsinki-NLP/opus-mt-en-ru"
    print(f"[*] Загрузка модели {model_name}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[*] Используется устройство: {device}")
    
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device)
    print("[*] Модель успешно загружена!\n")

    for cluster in os.listdir(base_dir):
        cluster_dir = os.path.join(base_dir, cluster)
        
        if not os.path.isdir(cluster_dir):
            continue
            
        print(f"\n{'='*40}\nPROCESSING CLUSTER: {cluster}\n{'='*40}")

        for index in range(1, 11):
            index_dir = os.path.join(cluster_dir, str(index))
            if not os.path.isdir(index_dir):
                continue
                
            my_input = index_dir      
            my_output = index_dir

            process_qa_answers_file(
                input_dir=my_input, 
                output_dir=my_output, 
                tokenizer=tokenizer, 
                model=model, 
                device=device, 
                batch_size=64  # Для MarianMT можно использовать большой батч
            )

[*] Загрузка модели Helsinki-NLP/opus-mt-en-ru...
[*] Используется устройство: cuda


/home/george/anaconda3/envs/hybrid_graphs/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


[*] Модель успешно загружена!


PROCESSING CLUSTER: pubs
  > Чтение файла: set/pubs/1/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/2/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/3/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/4/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/5/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/6/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/7/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/pubs/8/QA_russian.

    [V] Ответы переведены и сохранены в QA_russian.json
  > Чтение файла: set/pubs/10/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).

PROCESSING CLUSTER: northwind
  > Чтение файла: set/northwind/1/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/northwind/2/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/northwind/3/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/northwind/4/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/northwind/5/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/northwind/6/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/

    [V] Ответы переведены и сохранены в QA_russian.json
  > Чтение файла: set/NCAA/2/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/3/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/4/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/5/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/6/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/7/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/8/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/NCAA/9/QA_russian.j

...

    [V] Ответы переведены и сохранены в QA_russian.json
  > Чтение файла: set/Mondial/10/QA_russian.json
    Перевод 1 уникальных текстовых ответов...


    [V] Ответы переведены и сохранены в QA_russian.json

PROCESSING CLUSTER: Basketball_men
  > Чтение файла: set/Basketball_men/1/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/2/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/3/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/4/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/5/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/6/QA_russian.json
    Перевод 1 уникальных текстовых ответов...


    [V] Ответы переведены и сохранены в QA_russian.json
  > Чтение файла: set/Basketball_men/7/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/8/QA_russian.json
    [!] Текстовые ответы для перевода не найдены (только числа/даты или файл пуст).
  > Чтение файла: set/Basketball_men/9/QA_russian.json
    Перевод 1 уникальных текстовых ответов...


    [V] Ответы переведены и сохранены в QA_russian.json
  > Чтение файла: set/Basketball_men/10/QA_russian.json
    Перевод 1 уникальных текстовых ответов...


    [V] Ответы переведены и сохранены в QA_russian.json


In [3]:
import os
import pandas as pd

def compare_csv_folders(folder_orig, folder_trans, report_file="length_diff_report.csv"):
    """
    Сравнивает файлы в двух папках и находит ячейки с разницей в длине > 50%.
    """
    # Список файлов, которые есть в обеих папках
    files_orig = set(f for f in os.listdir(folder_orig) if f.endswith('.csv'))
    files_trans = set(f for f in os.listdir(folder_trans) if f.endswith('.csv'))
    common_files = files_orig.intersection(files_trans)

    if not common_files:
        print("[!] Нет общих CSV файлов для сравнения.")
        return

    results = []

    print(f"[*] Сравнение {len(common_files)} файлов...")

    for filename in common_files:
        path_a = os.path.join(folder_orig, filename)
        path_b = os.path.join(folder_trans, filename)

        try:
            df_a = pd.read_csv(path_a, keep_default_na=False)
            df_b = pd.read_csv(path_b, keep_default_na=False)

            # Проверка на одинаковую размерность
            if df_a.shape != df_b.shape:
                print(f"[!] Файл {filename} имеет разную структуру в папках. Пропуск.")
                continue

            for row_idx in range(len(df_a)):
                for col_idx in range(len(df_a.columns)):
                    val_a = str(df_a.iloc[row_idx, col_idx])
                    val_b = str(df_b.iloc[row_idx, col_idx])

                    len_a = len(val_a)
                    len_b = len(val_b)

                    # Пропускаем пустые или очень короткие строки (меньше 3 символов), 
                    # чтобы не спамить на разнице типа "No" -> "Нет"
                    if len_a <= 3:
                        continue

                    # Вычисляем разницу
                    # Условие: длина изменилась более чем на 50% от оригинала
                    # Например: было 100, стало 151 или 49.
                    diff_ratio = abs(len_a - len_b) / max(len_a, 1)

                    if diff_ratio > 2:
                        results.append({
                            'File': filename,
                            'Row': row_idx + 2, # +2 для соответствия номеру строки в Excel/CSV редакторах
                            'Column': df_a.columns[col_idx],
                            'Length_Orig': len_a,
                            'Length_Trans': len_b,
                            'Diff_%': round(diff_ratio * 100, 1),
                            'Original': val_a[:50] + "..." if len_a > 50 else val_a,
                            'Translated': val_b[:50] + "..." if len_b > 50 else val_b
                        })

        except Exception as e:
            print(f"[!] Ошибка при сравнении файла {filename}: {e}")

    # Вывод результатов
    if results:
        report_df = pd.DataFrame(results)
        report_df.to_csv(report_file, index=False, encoding='utf-8-sig')
        print(f"\n[!] Найдено подозрительных ячеек: {len(results)}")
        print(f"[V] Полный отчет сохранен в: {report_file}")
        
        # Печатаем первые 5 для примера
        print("\nПримеры значительных расхождений:")
        print(report_df[['File', 'Column', 'Diff_%', 'Original', 'Translated']].head().to_string())
    else:
        print("\n[OK] Значительных расхождений в длине строк не найдено.")

In [5]:
base_dir = "set/"

for cluster in os.listdir("set"):
    cluster_dir = os.path.join(base_dir, cluster)
    print(f"PROCESSING {cluster}")

    for index in range(1, 11):
        my_input = os.path.join(cluster_dir, str(index), "csv")
        my_output = os.path.join(cluster_dir, str(index), "csv_russian_2")

        compare_csv_folders(my_input, my_output, os.path.join(cluster_dir, str(index), "diff.csv"))

PROCESSING pubs
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 11 файлов...

[OK] Значительных расхождений в длине строк не найдено.
PROCESSING northwind
[*] Сравнение 30 файлов...

[OK] Значительных расхождений в длине строк не найдено.
[*] Сравнение 30 файлов...

[OK] Значит

In [1]:
import os
import pandas as pd
import re
import torch
from transformers import pipeline
from tqdm import tqdm

# --- ИНИЦИАЛИЗАЦИЯ NLLB ---
print("[*] Загрузка модели NLLB...")
# Проверяем, доступна ли видеокарта. 0 - первая GPU, -1 - процессор (CPU)
device = 0 if torch.cuda.is_available() else -1

# Инициализируем пайплайн перевода (600M - легкая и быстрая версия)
nllb_translator = pipeline(
    "translation", 
    model="facebook/nllb-200-distilled-600M", 
    src_lang="eng_Latn", # Английский
    tgt_lang="rus_Cyrl", # Русский
    device=device,
    max_length=512       # Максимальная длина ячейки
)
print(f"[*] Модель загружена на: {'GPU' if device == 0 else 'CPU'}")

# --- БАЗОВЫЕ ФУНКЦИИ ---

def is_id_column(col_name):
    return 'id' in col_name.lower()

def is_translatable(text):
    if not isinstance(text, str): return False
    text = text.strip()
    if not text or re.match(r'^[\d\s\.,\-:;/]+$', text): return False
    return True

def translate_texts_batch(texts, batch_size=32):
    """
    Переводит список строк пачками (батчами) для максимального ускорения.
    """
    if not texts:
        return []
    
    results =[]
    # Разбиваем список на пачки и переводим
    for i in tqdm(range(0, len(texts), batch_size), desc="Перевод батчей"):
        batch = texts[i : i + batch_size]
        # Пайплайн сам обрабатывает список строк
        out = nllb_translator(batch)
        results.extend([res['translation_text'] for res in out])
        
    return results

# --- ЛОГИКА ВАЛИДАЦИИ И ИСПРАВЛЕНИЯ ---

def fix_suspicious_translations(input_dir, output_dir, threshold=0.5):
    """
    Находит аномалии и переводит их заново с помощью NLLB.
    """
    files =[f for f in os.listdir(input_dir) if f.endswith('.csv')]
    bad_translations_map = {} 
    to_retranslate = set()    

    print(f"\n>>> ПРОВЕРКА КАЧЕСТВА (Модель: NLLB-200)...")

    # 1. Находим все уникальные подозрительные тексты
    for filename in files:
        orig_path = os.path.join(input_dir, filename)
        trans_path = os.path.join(output_dir, filename)
        
        if not os.path.exists(trans_path): continue

        df_orig = pd.read_csv(orig_path, keep_default_na=False)
        df_trans = pd.read_csv(trans_path, keep_default_na=False)

        for col in df_orig.columns:
            if is_id_column(col): continue
            
            for i in range(len(df_orig)):
                val_en = str(df_orig.iloc[i][col])
                val_ru = str(df_trans.iloc[i][col])
                
                if is_translatable(val_en):
                    len_en = len(val_en)
                    len_ru = len(val_ru)
                    
                    diff = abs(len_en - len_ru) / max(len_en, 1)
                    if diff > threshold and len_en > 3:
                        to_retranslate.add(val_en)

    if not to_retranslate:
        print("[OK] Аномалий не обнаружено.")
        return

    # Преобразуем set в list для передачи в батчи
    texts_en = list(to_retranslate)
    print(f"[*] Найдено уникальных подозрительных строк: {len(texts_en)}. Начинаем патчинг...")

    # 2. Массовый перевод подозрительных текстов одной командой
    # batch_size=32 оптимально для 8GB VRAM. Можно снизить до 16, если будет OOM (Out of memory)
    texts_ru = translate_texts_batch(texts_en, batch_size=4)

    # Заполняем словарь
    for en, ru in zip(texts_en, texts_ru):
        bad_translations_map[en] = ru

    # 3. Применяем исправления в файлы
    for filename in files:
        trans_path = os.path.join(output_dir, filename)
        orig_path = os.path.join(input_dir, filename)
        
        if not os.path.exists(trans_path): continue

        df_orig = pd.read_csv(orig_path, keep_default_na=False)
        df_trans = pd.read_csv(trans_path, keep_default_na=False)

        for col in df_trans.columns:
            if is_id_column(col): continue
            
            mask = df_orig[col].isin(bad_translations_map.keys())
            if mask.any():
                df_trans.loc[mask, col] = df_orig.loc[mask, col].map(bad_translations_map)
        
        df_trans.to_csv(trans_path, index=False, encoding='utf-8-sig')
    print("[V] Патчинг успешно завершен.")

# --- ОСНОВНОЙ ЦИКЛ ОБРАБОТКИ ---

def collect_unique_values(files, input_dir):
    unique_texts = set()
    for filename in files:
        try:
            df = pd.read_csv(os.path.join(input_dir, filename), keep_default_na=False)
            target_cols =[c for c in df.columns if not is_id_column(c)]
            for col in target_cols:
                for val in df[col].dropna().unique():
                    if is_translatable(val): unique_texts.add(val)
        except: pass
    return list(unique_texts)

def apply_translations(files, input_dir, output_dir, translation_map):
    os.makedirs(output_dir, exist_ok=True)
    for filename in files:
        df = pd.read_csv(os.path.join(input_dir, filename), keep_default_na=False)
        target_cols =[c for c in df.columns if not is_id_column(c)]
        for col in target_cols:
            df[col] = df[col].map(lambda x: translation_map.get(x, x))
        df.to_csv(os.path.join(output_dir, filename), index=False, encoding='utf-8-sig')

# --- ЗАПУСК ---

if __name__ == "__main__":
    base_dir = "set/"

    # Если папка base_dir не существует, прерываем, чтобы не было ошибки
    if not os.path.exists(base_dir):
        print(f"[!] Папка {base_dir} не найдена. Создайте её и поместите туда данные.")
        exit()

    for cluster in os.listdir(base_dir):
        cluster_dir = os.path.join(base_dir, cluster)
        
        # Защита от скрытых файлов (типа .DS_Store)
        if not os.path.isdir(cluster_dir): continue 
        
        for index in range(1, 11):
            my_input = os.path.join(cluster_dir, str(index), "csv")
            my_output = os.path.join(cluster_dir, str(index), "csv_russian")
            
            if not os.path.exists(my_input): continue
            
            files =[f for f in os.listdir(my_input) if f.endswith('.csv')]
            
            # --- РАСКОММЕНТИРОВАТЬ ДЛЯ ПОЛНОГО ЦИКЛА С NLLB ---
            # unique_texts = collect_unique_values(files, my_input)
            # texts_ru = translate_texts_batch(unique_texts, batch_size=32)
            # t_map = dict(zip(unique_texts, texts_ru))
            # apply_translations(files, my_input, my_output, t_map)
            
            # 2. Поиск аномалий и исправление (Патчинг)
            print(f"\n[Step 2] Validating and Patching {cluster}/{index} with NLLB")
            fix_suspicious_translations(my_input, my_output, threshold=0.5)

/home/george/anaconda3/envs/hybrid_graphs/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[*] Загрузка модели NLLB...


KeyboardInterrupt: 